# LandslideGuard - Stage 2: U-Net Training (Kaggle GPU)

Trains the 14-channel U-Net on Landslide4Sense using the frozen Stage-1
data pipeline and the train-only z-score normalization statistics from
`outputs/detection/data_verification/normalization_statistics.json`.

**Prerequisites (see notebook 02):**

- Kaggle GPU accelerator enabled.
- Landslide4Sense dataset attached (path -> `DATA_ROOT`).
- LandslideGuard repo reachable (URL -> `GITHUB_REPO`).

**What this notebook does:**

1. Clones the LandslideGuard repo and imports the frozen Stage-1 modules.
2. Builds train/valid/test DataLoaders (augmentation on train only).
3. Instantiates a from-scratch U-Net (14 in / 1 out), AdamW + cosine LR,
   BCE+Dice loss (0.5/0.5), AMP on CUDA.
4. Trains for `EPOCHS` epochs; saves the best-by-val-IoU checkpoint to
   `/kaggle/working/best_model.pth`.
5. Loads the best checkpoint and evaluates **once** on the test split.
6. Plots loss and metric curves; saves sample prediction overlays.

**What this notebook does not do:** Stage-1 preprocessing was already
verified; nothing here recomputes normalization statistics. The training
loop is standard supervised learning - no meta-learning, no self-supervision.


## 01 - Configuration

Edit `GITHUB_REPO` and `DATA_ROOT`. Everything else uses the values from
`configs/detection.yaml`.


In [ ]:
# ==== EDIT THESE TWO ====
GITHUB_REPO = "REPLACE-WITH-YOUR-REPO-URL"   # e.g. "https://github.com/<owner>/<repo>.git"
GITHUB_REF  = "main"
DATA_ROOT   = "/kaggle/input/landslide4sense"  # directory containing TrainData/ ValidData/ TestData/
# ========================

# You can also override a few knobs here (leave as None to use configs/detection.yaml).
OVERRIDE_EPOCHS       = None    # e.g. 5 for a quick sanity run
OVERRIDE_BATCH_SIZE   = None
OVERRIDE_LR           = None
OVERRIDE_BASE_FEATURES = None


## 02 - Environment + GPU


In [ ]:
import os, sys, subprocess, importlib
import torch

print("python:", sys.version.split()[0])
print("torch :", torch.__version__)
gpu_ok = torch.cuda.is_available()
if not gpu_ok:
    raise RuntimeError("GPU not available. Enable the accelerator in Session options.")
DEVICE = torch.device("cuda")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda[{i}] {p.name}  {p.total_memory/1024**3:.1f} GiB")
print("cuda   :", torch.version.cuda)


## 03 - Clone repo + set sys.path


In [ ]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORK / "LandslideGuard"

def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.stderr: print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"command failed with exit {r.returncode}")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
    run(["git", "checkout", GITHUB_REF],       cwd=REPO_DIR)
    run(["git", "pull", "--ff-only"],          cwd=REPO_DIR)
else:
    if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--depth=1", "--branch", GITHUB_REF, GITHUB_REPO, str(REPO_DIR)])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
head = subprocess.run(["git","rev-parse","HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
print("HEAD:", head)


## 04 - Load config + Stage-1 imports


In [ ]:
import yaml
cfg = yaml.safe_load((REPO_DIR / "configs" / "detection.yaml").read_text())
model_cfg = cfg["model"]
train_cfg = cfg["training"]
loss_cfg  = train_cfg["loss"]

if OVERRIDE_EPOCHS        is not None: train_cfg["epochs"]         = int(OVERRIDE_EPOCHS)
if OVERRIDE_BATCH_SIZE    is not None: train_cfg["batch_size"]     = int(OVERRIDE_BATCH_SIZE)
if OVERRIDE_LR            is not None: train_cfg["learning_rate"]  = float(OVERRIDE_LR)
if OVERRIDE_BASE_FEATURES is not None: model_cfg["base_features"]  = int(OVERRIDE_BASE_FEATURES)

from src.detection.preprocessing import NormalizationStats
from src.detection.dataset       import Landslide4SenseDataset, build_dataloader
from src.detection.models        import UNet, count_parameters
from src.detection.losses        import BCEDiceLoss
from src.detection.train         import Trainer, fit, evaluate, set_seed

print(json.dumps({"model": model_cfg, "training": train_cfg}, indent=2)
      if False else "config loaded.")
import json
print(json.dumps({"model": model_cfg, "training": train_cfg}, indent=2))


## 05 - Resolve dataset paths + build DataLoaders

Auto-detects the flat vs archive-nested layout. Uses the frozen Stage-1
train-only normalization statistics from the cloned repo - never recomputed.


In [ ]:
root = Path(DATA_ROOT)
assert root.is_dir(), f"DATA_ROOT does not exist: {root}"

def resolve_split(root, split):
    for img_d, mask_d in [(root / split / "img",         root / split / "mask"),
                          (root / split / split / "img", root / split / split / "mask")]:
        if img_d.is_dir() and mask_d.is_dir():
            return img_d, mask_d
    raise FileNotFoundError(f"could not find img/mask under {root/split}")

paths = {name: resolve_split(root, name)
         for name in ["TrainData", "ValidData", "TestData"]}

stats = NormalizationStats.from_json(
    REPO_DIR / "outputs" / "detection" / "data_verification"
    / "normalization_statistics.json")

ds_train = Landslide4SenseDataset(*paths["TrainData"], stats=stats,
                                  split="train", augment_seed=train_cfg["seed"])
ds_valid = Landslide4SenseDataset(*paths["ValidData"], stats=stats, split="valid")
ds_test  = Landslide4SenseDataset(*paths["TestData"],  stats=stats, split="test")

for name, d in [("train",ds_train), ("valid",ds_valid), ("test",ds_test)]:
    print(f"{name:5s} n={len(d):5d}  augment={d._aug is not None}")

nw = train_cfg.get("num_workers", 2)
pm = bool(train_cfg.get("pin_memory", True))
B  = int(train_cfg["batch_size"])
train_loader = build_dataloader(ds_train, batch_size=B, num_workers=nw, pin_memory=pm)
valid_loader = build_dataloader(ds_valid, batch_size=B, num_workers=nw, pin_memory=pm)
test_loader  = build_dataloader(ds_test,  batch_size=B, num_workers=nw, pin_memory=pm)


## 06 - Model + loss + optimizer + scheduler


In [ ]:
set_seed(train_cfg["seed"])

model = UNet(in_channels=int(model_cfg["in_channels"]),
             out_channels=int(model_cfg["out_channels"]),
             base_features=int(model_cfg["base_features"]))
print(f"U-Net params: {count_parameters(model):,}")

loss_fn = BCEDiceLoss(bce_weight=float(loss_cfg["bce_weight"]),
                      dice_weight=float(loss_cfg["dice_weight"]),
                      dice_eps=float(loss_cfg.get("dice_eps", 1.0)))

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=float(train_cfg["learning_rate"]),
                              weight_decay=float(train_cfg["weight_decay"]))

sched_name = train_cfg.get("scheduler", "cosine")
if sched_name == "cosine":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=int(train_cfg["epochs"]))
elif sched_name in (None, "none", "off"):
    scheduler = None
else:
    raise ValueError(f"unknown scheduler: {sched_name}")

trainer = Trainer(model=model, loss_fn=loss_fn, optimizer=optimizer,
                  device=DEVICE, scheduler=scheduler,
                  use_amp=bool(train_cfg.get("amp", True)),
                  grad_clip=train_cfg.get("grad_clip", None),
                  threshold=float(cfg["inference"]["threshold"]))


## 07 - One-batch sanity check (before training)


In [ ]:
xb, yb = next(iter(train_loader))
xb = xb.to(DEVICE); yb = yb.to(DEVICE)
model.eval()
with torch.no_grad():
    logits = model(xb)
    l0 = loss_fn(logits, yb).item()
print("batch  x:", tuple(xb.shape), "y:", tuple(yb.shape), "device:", xb.device)
print("logits:", tuple(logits.shape),
      f"range=({logits.min().item():+.3f}, {logits.max().item():+.3f})")
print(f"initial BCE+Dice loss = {l0:.4f}")


## 08 - Train

Runs `epochs` epochs, keeps the best-val-IoU checkpoint at
`/kaggle/working/best_model.pth`, and streams a JSON history log to
`/kaggle/working/history.json`.


In [ ]:
out_dir = WORK
checkpoint_path = out_dir / "best_model.pth"
history_path    = out_dir / "history.json"

history = fit(trainer, train_loader, valid_loader,
              epochs=int(train_cfg["epochs"]),
              checkpoint_path=checkpoint_path,
              history_path=history_path,
              select_by=str(train_cfg.get("select_by", "val_iou")))

print()
print("best checkpoint :", checkpoint_path)
print("history         :", history_path)


## 09 - Loss + metric curves


In [ ]:
import matplotlib.pyplot as plt

ep = [e.epoch for e in history.epochs]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, [e.train_loss for e in history.epochs], label="train")
axes[0].plot(ep, [e.val_loss   for e in history.epochs], label="val")
axes[0].set_title("BCE+Dice loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(ep, [e.val_iou for e in history.epochs], label="val IoU")
axes[1].plot(ep, [e.val_f1  for e in history.epochs], label="val F1")
axes[1].plot(ep, [e.val_precision for e in history.epochs], label="val P", alpha=0.6)
axes[1].plot(ep, [e.val_recall    for e in history.epochs], label="val R", alpha=0.6)
axes[1].set_title("Validation metrics"); axes[1].set_xlabel("epoch"); axes[1].legend()
fig.tight_layout()
fig.savefig(out_dir / "curves.png", dpi=120)
plt.show()


## 10 - Load best checkpoint and evaluate on the test split (once)

The test split is evaluated exactly one time, with the best-val-IoU
checkpoint. Do not iterate on this number - it is your final Stage-2
metric.


In [ ]:
ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
print(f"loaded checkpoint from epoch {ckpt['epoch']}; "
      f"val metrics at save time: iou={ckpt['val_metrics']['iou']:.4f} "
      f"f1={ckpt['val_metrics']['f1']:.4f}")

test_metrics = evaluate(model, test_loader, DEVICE,
                        threshold=float(cfg["inference"]["threshold"]))
print()
print("== TEST-SET METRICS ==")
for k in ("iou","f1","precision","recall","tp","fp","fn","tn","threshold"):
    print(f"  {k:10s} {test_metrics[k]}")

import json
(out_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))


## 11 - Sample prediction overlays

Saves 6 sample overlays from the test split (image RGB + ground truth
+ prediction) to `/kaggle/working/predictions/`.


In [ ]:
import numpy as np
from pathlib import Path
pred_dir = out_dir / "predictions"
pred_dir.mkdir(exist_ok=True)

def pct(x, lo=2, hi=98):
    a, b = np.percentile(x, lo), np.percentile(x, hi)
    if b <= a: return np.zeros_like(x)
    return np.clip((x - a) / (b - a), 0, 1)

model.eval()
seen = 0
max_samples = 6
with torch.no_grad():
    for xb, yb in test_loader:
        xb_dev = xb.to(DEVICE); yb_dev = yb.to(DEVICE)
        prob = torch.sigmoid(model(xb_dev))
        for i in range(xb.shape[0]):
            if seen >= max_samples: break
            img = xb[i].cpu().numpy()  # (14, H, W) - already z-scored
            # Use raw dataset for viewable RGB (bands 3,2,1 are Red,Green,Blue)
            rgb = np.stack([pct(img[3]), pct(img[2]), pct(img[1])], axis=-1)
            m_true = yb[i].cpu().numpy()
            m_pred = (prob[i, 0].cpu().numpy() >= float(cfg["inference"]["threshold"])).astype(np.uint8)
            fig, ax = plt.subplots(1, 3, figsize=(11, 4))
            ax[0].imshow(rgb); ax[0].set_title("RGB")
            ax[1].imshow(m_true, cmap="gray", vmin=0, vmax=1); ax[1].set_title("ground truth")
            ax[2].imshow(m_pred, cmap="gray", vmin=0, vmax=1); ax[2].set_title(f"prediction @ {float(cfg['inference']['threshold'])}")
            for a in ax: a.set_xticks([]); a.set_yticks([])
            fig.tight_layout()
            fig.savefig(pred_dir / f"sample_{seen:02d}.png", dpi=120)
            plt.close(fig)
            seen += 1
        if seen >= max_samples: break
print(f"wrote {seen} sample overlays to {pred_dir}")


## 12 - Summary

Artifacts written under `/kaggle/working/`:

| Path | What |
|------|------|
| `best_model.pth` | best-val-IoU U-Net checkpoint |
| `history.json`   | per-epoch train/val loss + metrics |
| `curves.png`     | training curves |
| `test_metrics.json` | final one-shot test metrics |
| `predictions/`   | 6 sample overlays |

To pull these down: **File -> Download outputs** in the Kaggle UI, or
attach `/kaggle/working/best_model.pth` to a follow-up notebook.

### What was NOT changed

- Stage-1 preprocessing (`src/detection/preprocessing.py`, `dataset.py`) is
  frozen. This notebook only *uses* it via `import`.
- The normalization statistics were loaded, not recomputed.
- The official Landslide4Sense train / valid / test split was preserved.
